## LTBI model to simulate results Gil et al.
    Marta Llabrés

#### Comments

- We are simulating tuberculosis in minipigs
- This new model is based on:
    - **Lesion**: evolution through a diffusion-reaction equation
    - **Fibroblasts**: evolution through both a chemokinetic and chemotactic term, and a proliferation term. The first one is proportional to the gradient of the lesion and is what drives the speed of the fibroblasts. The second one drives the direction of the movement (in principle) to tighten the encapsulation ring. The third one is to model the action of the TGF-$\beta$, and it is proportional to the gradient of the lesion.
    - **Calcification**: the necrosis of the lesion starts after encapsulation is completed, and no more oxygen nor nutrients can access within. The idea is that necrosis starts at the core of the lesion and expands towards the outsides. We have modelled it with an initial seed at the centre and then a reaction-diffusion equation. Keep in mind that the diffusion term does not model a *real* diffusion, but mimicks the growth of necrosis.

  The ODEs look like:

  $$
\partial_t c_l = k_l g c_l (c_l-a) + \nabla \left( D_l g \nabla c_l \right) - k_{cal} c_{cal} c_l
$$

$$
\partial_t c_f = k_f g |\nabla c_l| c_f + \nabla \left[ \chi_{chem} g |\nabla c_l| \nabla c_f  - \chi_{tax} g  c_f \nabla c_l \right]
$$

$$
\partial_t c_{cal} = k_{cal} c_{cal} c_l + \nabla \left( D_{cal}  \nabla c_{cal} \right)
$$


**Goal**
We want to mimic the results obtained in the research by Gil et al. [1].
For that we will eliminate random lesion proliferation and set it to predefined times. New lesion origins are also predetermined. We will also add the necessary septae to ensure correct encapsulation.

#### Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v2 as imageio
import os
import pickle
from scipy.stats import qmc
from shapely.geometry import Polygon, Point, box
from scipy.spatial import Voronoi, voronoi_plot_2d
from shapely import wkt, affinity
from typing import List, Tuple, Dict
from scipy.optimize import minimize
import statistics as stat
from scipy.ndimage import distance_transform_edt
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Rectangle
from matplotlib.patches import Patch
import matplotlib as mpl

# ------- let's import our evolution function ----
import model_evol_PJ_new_cal as mfun5

#### Functions

In [ ]:
def create_granuloma(state, calcification, c_f1, e):
    mask = (state[f'c_l_{e}'] + calcification[f'c_cal_{e}']) > 0.01
    
    # compute the distance of outward pixels from the lesion
    dist = distance_transform_edt(~mask)
    
    # let's define our interest band
    inner = 0    
    outer = 6      
    
    crown = (dist >= inner) & (dist <= outer)
    granuloma = crown * c_f1 + calcification[f'c_cal_{e}'] + state[f'c_l_{e}']
    lesion = calcification[f'c_cal_{e}'] + state[f'c_l_{e}']
    
    return granuloma, lesion

In [ ]:
def calcul_radius_cells(m, dx):  
    return np.sqrt((dx**2)* np.sum(m > 0.05)/ (3.14))

def calcul_radius(m, dx):  
    return np.sqrt((dx**2)* np.sum(m[m > 0.05])/ (3.14)) 

#### Parameters

In [ ]:
# ======= FIXED PARAMETERS ========
    # space-time discretization
dx = 0.05  # [mm] 
lx = 180
ly = 150
dt = 0.00025 # [days]
iter = 200000

    #  for lesion initialization
a = 0.14 # g/mm3   threshold for lesion growth
eps = 1 # lesion sharpness
r0 = 1.5 # lesion initial radius in terms of grid cells
alfa = 1 # belongs to [0,1]. initial_concentration of lesion
count = 1 # initial number of lesions
age1 = 0 # initial age of lesion 1 (in days!!!)


# we create a dictionary with the fixed parameters
params_fixed = {"a": a, "eps": eps, "r0":r0, "alfa": alfa, "count": count, "dx": dx, "lx": lx, "ly": ly, "dt": dt, "age1": age1} 

# ======= ESTIMATED PARAMETERS ========
k = 9 # lesion reaction coefficient k_l
D_l = 5e-3 # lesion diffusion coefficient D_l
xi_kin = 5*0.9e-1 # coefficient of chemokinetic term for fibroblasts # 10*0.9e-1
xi_tax = 15*0.9e-1 # coefficient of chemotactic term for fibroblasts
k_f = 8*3 # fibroblasts reaction / proliferation coefficient
k_cal = 9 # calcification reaction coefficient
D_cal = .2e-2  # calcification "diffusion" coefficient
#-----------
k_nuc = 0.0035 
k_grow = 2.5
dcal = 0.001
#----------
rho = 0.00120/10 #fitting parameter
alpha = 0.034638
n = 1.63483


param_names = ["k", "D", "xi", "xi_tax", "k_f", "k_nuc", "k_grow", "dcal", "rho", "alpha", "n"] 
best_params = [k, D_l, xi_kin, xi_tax, k_f, k_nuc, k_grow, dcal, rho, alpha, n]

#### Geometric initialization

In [ ]:
# first, let's load our secondary lobule and origins of lesions 
with open("central_region2.pkl", "rb") as f:
    central_region = pickle.load(f)

with open("origins.pkl", "rb") as f:
    origins_point = pickle.load(f)  
    

In [ ]:
x, y = central_region.exterior.xy
origins = [(o.x, o.y) for o in origins_point]
born_steps = [59503, 71442, 92704, 101481, 112604] # deterministic birth times

origin1 = origins[0]
params_fixed["origin1"] = origin1 
params_fixed["origins"] = origins
params_fixed["central_region"] = central_region
params_fixed["born_steps"] = born_steps

# let's plot the full region
x_poly = np.array(x) * dx
y_poly = np.array(y) * dx

fig, ax = plt.subplots()
ax.plot(x_poly, y_poly, 'k-', linewidth=1)
ax.fill(x_poly, y_poly, color='darkgray', alpha=0.5)
for e in range (0, 6):
    origin = np.array(origins[e]) * dx
    ax.scatter(origin[0], origin[1], label=f'origin_{e+1}')
for interior in central_region.interiors:
    xi, yi = interior.xy
    xi, yi = np.array(xi) * dx, np.array(yi) * dx

    ax.plot(xi, yi, '-k')
    ax.fill(xi, yi, color='white')

plt.xlabel(r'$x$ (mm)')
plt.ylabel(r'$y$ (mm)')
#filename = f"pj_area.pdf"
#plt.savefig(filename, format='pdf', dpi=300) 
plt.show()

#### Execution of program

In [ ]:
# let's join the parameters
params = dict(zip(param_names, best_params))
params.update(params_fixed)
print(params.keys())

In [ ]:
# let's execute the full program
state, ages, origins, radius, radius_lc, full_radius, calcification, c_f1, max_grad, DIF_kin_vector, DIF_tax_vector, prol_vector, cal_dif_vec, probs, phases, i_encaps, encapsulation, roots = mfun5.model_evolution_loop(250000, params)

#### Results and plots

In [ ]:
x, y = central_region.exterior.xy
x_poly = np.array(x) * dx
y_poly = np.array(y) * dx

# heatmaps for the variables
cmap_cl = plt.cm.YlOrRd    
cmap_cf = plt.cm.YlGnBu   
cmap_ccal = LinearSegmentedColormap.from_list('ccal', ['#ffffe5', '#737373', '#000000'])


# latex style configuration
mpl.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern Roman"], "axes.labelsize": 12, "font.size": 11, "legend.fontsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10})

# phases visualization
# --- colour map for each phase ---
phase_colors = {0:'#ffffff', 1:'#fff3cd', 2:'#ffd6a5', 3:'#cce5ff', 4:'#d4edda'}
phase_labels = {0: 'Disappeared', 1: 'Phase I: Growing', 2: 'Phase II: Encapsulating', 3: 'Phase III: Calcifying', 4: 'Phase IV: Resolved'}


In [ ]:
# data organization
c_l = state["c_l_1"].T 
c_f = c_f1.T
c_cal = calcification["c_cal_1"].T

    # radius evolution variables
c_tot = c_l + c_f + c_cal
rad_les = radius["r_1"]
rad_calc = radius_lc["r_lc_1"]
full_radius = full_radius["full_r_1"]

time = np.arange(iter) * dt
extent_full = [0, c_l.shape[1]*dx, 0, c_l.shape[0]*dx]


In [ ]:
c_l = state["c_l_1"].T
total_c_l = sum(state.values())
c_cal = sum(calcification.values())
granulomas = total_c_l + c_cal + c_f1
extent_full = [0, c_l.shape[1]*dx, 0, c_l.shape[0]*dx]


# radius evolution plot + background phases
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) lesion density
ax_cl = axes[0]
im_cl = ax_cl.imshow(total_c_l.T, cmap=cmap_cl, origin='lower', extent=extent_full, interpolation='bilinear', vmin = 0, vmax = 1)
ax_cl.plot(x_poly, y_poly, color='black', linewidth=.75, label='Lobule Boundary')
ax_cl.set_xlabel(r'$x$ (mm)')
ax_cl.set_ylabel(r'$y$ (mm)')
#ax_cl.set_title(r'(a)', loc='left')
divider = make_axes_locatable(ax_cl)
cax_cl = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cl, cax=cax_cl, label=r'$c_l$')

# 2) fibroblast density
ax_cf = axes[1]
im_cf = ax_cf.imshow(c_f1.T, cmap=cmap_cf, origin='lower', extent=extent_full, interpolation='bilinear', vmin = 0, vmax = 1)
ax_cf.plot(x_poly, y_poly, color='black', linewidth=.75, label='Lobule Boundary')
ax_cf.set_xlabel(r'$x$ (mm)')
ax_cf.set_ylabel(r'$y$ (mm)')
#ax_cf.set_title(r'(b)', loc='left')
divider2 = make_axes_locatable(ax_cf)
cax_cf = divider2.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cf, cax=cax_cf, label=r'$c_f$')

# 3) calcification density
ax_ccal = axes[2]
cal_max = np.max(c_cal) if np.max(c_cal) > 1e-6 else 1.0 
im_ccal = ax_ccal.imshow(c_cal.T, cmap=cmap_ccal, origin='lower', extent=extent_full, interpolation='bilinear', vmin=0, vmax=1)
ax_ccal.plot(x_poly, y_poly, color='black', linewidth=.75, label='Lobule Boundary')
ax_ccal.set_xlabel(r'$x$ (mm)')
ax_ccal.set_ylabel(r'$y$ (mm)')
#ax_ccal.set_title(r'(c)', loc='left')
divider3 = make_axes_locatable(ax_ccal)
cax_ccal = divider3.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_ccal, cax=cax_ccal, label=r'$c_{cal}$')

plt.tight_layout()
plt.savefig("model_PJ.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
colors_fibro = ["#EAEBED", "#D6D2D8", "#D0C8D2", "#C9BDCB", "#CDC3CF", "#C6B8C8", "#C5ACC9"]
colors_lesion = ["#B79EC0", "#AC94B8", "#A487B1", "#9E7BAA", "#956FA3", "#8F6799"]
colors_cal = ["#83588C", "#7C5085", "#633968", "#49254B"]

combined_colors = colors_fibro + colors_lesion + colors_cal
tb_cmap = LinearSegmentedColormap.from_list("tb_map", combined_colors, N=256)

def normalize(c):
    c = c - np.min(c)
    return c / (np.max(c) + 1e-12)

f = normalize(c_f1)
l = normalize(total_c_l)
k = normalize(c_cal)
wf, wl, wk = 1.3, 3.0, 5.0
signal = wf * f + wl * l + wk * k


signal = signal - np.min(signal)
signal = signal / (np.max(signal) + 1e-12)

ref_img = mpimg.imread('poly_scale.png')
ref_h, ref_w = ref_img.shape[:2]
ref_aspect = ref_w / ref_h  

fig_height = 5  
fig_width = fig_height * ref_aspect


fig, ax = plt.subplots(1, 2, figsize=(fig_width, fig_height))
ax_t = ax[0]
ax_t.imshow(ref_img)
ax_t.axis("off")
ax_t.set_title(r'(a)', loc='left')



ax_cl = ax[1]
im = ax_cl.imshow(signal.T, cmap=tb_cmap, origin='lower')
ax_cl.axis('off')
ax_cl.set_title(r'(b)', loc='left')

divider = make_axes_locatable(ax_cl)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)

cbar.set_ticks([0.0, 0.33, 0.66, 0.9])
cbar.set_ticklabels([
    "Fibrotic tissue",
    "Early lesion",
    "Lesion",
    "Calcified core"
])

filename = "result_PJ_stain.pdf"
fig.savefig(filename, format='pdf', dpi=300, bbox_inches='tight')
plt.show()

#### References

[1] O. Gil, I. Díaz, C. Vilaplana, G. Tapia, J. Díaz, M. Fort, N. Cáceres, S. Pinto, J. Caylà, L. Corner, M. Domingo, and P.-J. Cardona, "Granuloma Encapsulation Is a Key Factor for Containing Tuberculosis Infection in Minipigs," *PLOS ONE*, vol. 5, no. 4, p. e10030, Apr. 2010. doi: 10.1371/journal.pone.0010030.